# trl-for-angels: Kaggle runner

The only notebook you need. Each run it pulls the project from GitHub, runs the task in
`kaggle/run.sh` (Claude picks it), and pushes the results to the `kaggle-results` branch,
where Claude reads them. You don't need to watch it.

**One-time setup**
1. Settings → **Accelerator: GPU T4** and **Internet: On**.
2. Add-ons → **Secrets**, attached to this notebook:
   - `GITHUB_TOKEN`: a fine-grained GitHub token for `Sidi3355/trl-for-angels` only, with
     **Contents: Read and write**.
   - `ANTHROPIC_API_KEY`: your Claude Console key (needed from step A4).

**Each run:** **Save Version → Save & Run All (Commit) → Save**. It runs in the background
on Kaggle; close the tab if you like. Then tell Claude it's started.

In [ ]:
# Secrets from Add-ons > Secrets, into this kernel's environment (never printed).
import os
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
for name in ("GITHUB_TOKEN", "ANTHROPIC_API_KEY"):
    try:
        os.environ[name] = secrets.get_secret(name).strip()  # a pasted newline breaks a token
        print(f"{name}: attached")
    except Exception:
        print(f"{name}: NOT attached to this notebook")
assert os.environ.get("GITHUB_TOKEN"), "Attach the GITHUB_TOKEN secret (Add-ons > Secrets)"

In [ ]:
%%bash
# Checks the GitHub token, clones the project, and hands over to kaggle/runner_main.sh.
set -euo pipefail
export REPO_URL=https://github.com/Sidi3355/trl-for-angels.git RESULTS_BRANCH=kaggle-results GIT_TERMINAL_PROMPT=0
export AUTH="AUTHORIZATION: basic $(printf 'x-access-token:%s' "$GITHUB_TOKEN" | base64 -w0)"
code=$(curl -s -o /dev/null -w '%{http_code}' -H "Authorization: Bearer $GITHUB_TOKEN" \
  https://api.github.com/repos/Sidi3355/trl-for-angels)
case "$code" in
  200) echo "GitHub token: OK, it can see the repo" ;;
  401) echo "GitHub token REJECTED (401): wrong, expired or pasted with extra characters." \
         "Make a new one and update the GITHUB_TOKEN secret."; exit 1 ;;
  403|404) echo "GitHub token can't see Sidi3355/trl-for-angels ($code): edit the token so" \
         "its repository access includes that repo, with Contents: Read and write."; exit 1 ;;
  *) echo "GitHub API check returned $code; trying the clone anyway" ;;
esac
mkdir -p /tmp/work && cd /tmp/work && rm -rf repo
git -c http.extraHeader="$AUTH" clone -q --depth 1 -b claude/determined-albattani-dl2cyy "$REPO_URL" repo
bash repo/kaggle/runner_main.sh